# Pareto fronts of pose priors: sample quality against the cost of sampling (proof of concept)

The models of `posedflow_vs_rsgm.ipynb` (PoseD-Flow, DPoser, RSGM on the data of PoseD-Flow) and the HuProSO3 flow, placed
as the MLIP papers place their potentials (e.g. Fig. 1 of the GRACE paper): one panel per metric, the quality on x
(the better side on the left), the time per pose on y (log scale), and the Pareto-optimal models joined by a line.

- **Quality.** Read from the cache of `posedflow_vs_rsgm.ipynb` (`~/data/experiments/posedflow_comparison/cache`), not
  computed again:
  - **Wasserstein** (our harness): optimal transport between 2,000 generated and 2,000 test poses, degrees per joint;
  - **FID** (protocol of PoseD-Flow): Fréchet distance of the 22 SMPL joint positions against their statistics
    `gt_mean_var_v2.npz`, 20 sets of 500 samples;
  - **APD** (protocol of PoseD-Flow): mean pairwise joint distance, cm. Their table reads it as "higher is better";
    it is a diversity, so the value of real poses is drawn as a reference line.

  A model missing from that cache (the HuProSO3 flow) is evaluated here with the same settings and written to the same
  cache, so `posedflow_vs_rsgm.ipynb` reuses it.
- **Cost.** The wall-clock time of `model.sample(n)` (noise → valid rotations, all sampler steps),
  median of `REPEATS` runs after a warm-up, with the device synchronized. Two regimes, as ASE / LAMMPS in GRACE:
  - **throughput**: n = 4096 (filled markers);
  - **latency**: n = 1 (open markers).

  Checkboxes above the plots choose the n drawn; a Pareto front is drawn for each.

  The times are cached next to the metrics, one file per chip and device
  (`runtime_<chip>_<device>.json`), with the chip, device, platform and torch version in the settings. The file
  keeps every n timed so far (`BATCHES` adds the missing ones); a file of another torch is set aside as
  `runtime_<chip>_<device>.torch_<version>.json`, not overwritten. Times of other machines (e.g. a GPU server, with
  `scripts/evaluate_pareto.py`) are read from the cache: a selector above the plots chooses the measurement, and
  the plots name the machine, torch, n and repeats.
- **Shared code.** The models (`MODELS`), the settings, the cache and the timing are in `hp.evaluation.pareto`;
  `scripts/evaluate_pareto.py` evaluates and times the same models on another machine (e.g. a GPU server) into the same
  cache. Edit `MODELS` there, and choose here which of them to use.
- **† = trained on our AMASS** (SMPL-X G), evaluated on the data of PoseD-Flow: a data shift that the others do not have.
  The HuProSO3 flow is the only such model here. It is drawn as a reference for the speed of a one-pass sampler (a
  diamond), and is left out of the Pareto fronts.

In [1]:
import platform
import sys
from pathlib import Path

# the code of the checkout this notebook is in (e.g. a worktree), before an editable install of another one
SRC = next(str(p / "src") for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hp").is_dir())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
import torch

from hp.evaluation import pareto as P
from hp.evaluation.data import load_reference

assert sys.modules["hp.evaluation.pareto"].__file__.startswith(SRC), \
    f"hp comes from {sys.modules['hp.evaluation.pareto'].__file__}: restart the kernel"

# ---- parameters ----------------------------------------------------------------------
# The models, the data and the evaluation settings are those of hp.evaluation.pareto (shared with
# scripts/evaluate_pareto.py, and the settings of posedflow_vs_rsgm.ipynb): edit MODELS there.
MODELS = P.select(None)  # all; or e.g. P.select(["RSGM", "DPoser"]) (names, short labels, or parts of them)

# Drawn, but left out of the Pareto fronts: trained on other data than the models it would be compared with
FRONT_EXCLUDE = {"HuProSO3 flow: our AMASS"}

# the families, in a fixed order of colours; RSGM (ours) in red, as GRACE in its figure
FAMILIES = {"RSGM": "#e34948", "PoseD-Flow": "#2a78d6", "DPoser": "#7b3fa0", "HuProSO3": "#1baf7a"}

DEVICE = P.default_device()
BATCHES = P.BATCHES  # the n to time (e.g. (1, 64, 1024, 4096, 16384)); the cache keeps every n timed before
SHOW_BATCHES = P.BATCHES  # the n checked at first in the plots (the checkboxes offer every n of a measurement)
REPEATS = P.REPEATS  # timed runs per n
MEASURE_HERE = True  # time the models on this machine (only what is not in the cache yet); False: only read the cache
FIGURES = P.CACHE.parent / "figures"  # the default folder of the save button
RECOMPUTE = set()  # names of models (or "<name> / <run>") to evaluate again (Wasserstein, FID, APD); True: all
RECOMPUTE_RUNTIME = set()  # names of models (or "<name> / <run>") to time the n of BATCHES again; True: all
# --------------------------------------------------------------------------------------
print(f"device {DEVICE}, {platform.platform()}, torch {torch.__version__}, cache {P.CACHE}")

device mps, macOS-26.6-arm64-arm-64bit, torch 2.13.0, cache /Users/evgeniifedorov/data/experiments/posedflow_comparison/cache


## 1. The models and the cache

The helpers of `hp.evaluation.pareto` (as in `posedflow_vs_rsgm.ipynb`): a model with several runs (seeds) is evaluated
per run; a result is reused only if it was computed with the same settings. A run that is not on this machine is
skipped.

In [2]:
zoo = P.Zoo(MODELS, DEVICE)
builders, identities, groups, group_of = zoo.builders, zoo.identities, zoo.groups, zoo.group_of

## 2. Quality: from the cache, or evaluated with the settings of `posedflow_vs_rsgm.ipynb`

The data is loaded as there (it sets the sizes in the cache settings). Only a model without a cached result is
evaluated (here: the HuProSO3 flow).

In [3]:
test = load_reference("test", P.ROTATIONS, "posedflow")
train = load_reference("train", P.ROTATIONS, "posedflow")
print(f"test {len(test):,}   train {len(train):,}")

harness, paper = P.evaluate_quality(zoo, test, train, RECOMPUTE)

test 54,200   train 1,861,105


## 3. Cost: the time of `model.sample(n)`

One run per model is timed (the seeds of a model have the same network and sampler). Batch 1 of a 1000-step sampler
takes 1000 network calls of a single pose: latency, not throughput. The times of this machine are measured here (what
the cache lacks); those of other machines are read from the cache.

In [4]:
if MEASURE_HERE:
    print(f"timing on {P.chip_name(DEVICE)} ({DEVICE}), saved as {P.runtime_stage(DEVICE)}.json")
    P.measure_runtime(zoo, RECOMPUTE_RUNTIME, BATCHES, REPEATS)

MEASUREMENTS = P.runtime_stages(zoo)  # {stage: (runtime[name], info)}: every machine in the cache
for stage, (rt, info) in MEASUREMENTS.items():
    print(f"{stage}: {len(rt)} of {len(groups)} models, torch {', '.join(info['torch'])}, repeats {', '.join(info['repeats'])}"
          + (f", measured {' to '.join(dict.fromkeys(info['dates']))}" if info["dates"] else ""))

timing on Apple M5 Pro (mps), saved as runtime_Apple_M5_Pro_mps.json
runtime_Apple_M5_Pro_mps: 12 of 12 models, torch 2.13.0, repeats 3
runtime_NVIDIA_GeForce_RTX_4090_cuda: 12 of 12 models, torch 2.14.1+cu130, repeats 5, measured 2026-11-10


## 4. The table

Of one measurement, at the n of `SHOW_BATCHES` (the selector in section 5 shows the table of its choice too).

In [5]:
def family(name):
    return next(f for f in FAMILIES if name.startswith(f))


def mean_over_runs(name, get):
    return float(np.mean([get(r) for r in (results[l] for l in groups[name])]))


results = {l: {"harness": harness[l], "paper": paper[l]} for l in builders}
METRICS = {  # name: (value of one run's results, unit, lower is better)
    "Wasserstein": (lambda r: r["harness"]["wasserstein"]["generated"]["joint_mean_deg"]["mean"], "deg / joint", True),
    "FID": (lambda r: r["paper"]["FID"]["mean"], "", True),
    "APD": (lambda r: r["paper"]["APD"]["mean"], "cm", False),
}
REAL = {  # real train poses, the yardstick of each metric
    "Wasserstein": next(iter(harness.values()))["wasserstein"]["contrast"]["joint_mean_deg"]["mean"],
    "FID": paper["real train poses"]["FID"]["mean"],
    "APD": paper["real train poses"]["APD"]["mean"],
}


def machine(info):
    """The machine and the settings of a measurement, in one line."""
    platform_ = ", ".join(dict.fromkeys(p.split("-")[0] + (" x86_64" if "x86_64" in p else " arm64" if "arm64" in p else "")
                                        for p in info["platform"]))
    when = f", measured {' to '.join(dict.fromkeys(info['dates']))}" if info["dates"] else ""
    return (f"{', '.join(info['chip'])} ({', '.join(info['device'])}), {platform_}, torch {', '.join(info['torch'])}; "
            f"median of {', '.join(info['repeats'])} runs after a warm-up{when}")


def timed_batches(runtime):
    """Every n of a measurement, the largest first."""
    return sorted({int(n) for rt in runtime.values() for n in rt["batches"]}, reverse=True)


def make_table(runtime, batches):
    """The rows of the models of `runtime`: times[n] in us per pose for the n of `batches` they are timed at."""
    table = []
    for name in groups:
        rt = runtime.get(name)
        if rt is None:
            print(f"{name}: not in this measurement, left out")
            continue
        times = {n: rt["batches"][str(n)]["us_per_pose"] for n in batches if str(n) in rt["batches"]}
        table.append(dict(name=name, short=MODELS[name]["short"], family=family(name), runs=len(groups[name]),
                          params_M=rt["parameters"] / 1e6, steps=rt["sample_steps"], times=times,
                          **{m: mean_over_runs(name, get) for m, (get, _, _) in METRICS.items()}))
    return table


def print_table(table, batches, info):
    columns = "".join(f" {f'n={n}':>10s}" for n in batches)
    print(f"{'model':24s} {'params M':>9s} {'steps':>6s}{columns} {'Wass.':>7s} {'FID':>7s} {'APD':>6s}")
    first = batches[0] if batches else None
    for r in sorted(table, key=lambda r: r["times"].get(first, np.inf)):
        times = "".join(f" {r['times'][n]:10.1f}" if n in r["times"] else f" {'-':>10s}" for n in batches)
        print(f"{r['short']:24s} {r['params_M']:9.2f} {str(r['steps']):>6s}{times} "
              f"{r['Wasserstein']:7.2f} {r['FID']:7.4f} {r['APD']:6.2f}")
    print(f"{'real train poses':24s} {'':9s} {'':6s}{' ' * 11 * len(batches)} {REAL['Wasserstein']:7.2f} {REAL['FID']:7.4f} {REAL['APD']:6.2f}")
    print(f"n=...: us per pose of sample(n). {machine(info)}.")


# the table of this machine if it is measured, else of the measurement with the most models
DEFAULT_MEASUREMENT = P.runtime_stage(DEVICE) if P.runtime_stage(DEVICE) in MEASUREMENTS else next(iter(MEASUREMENTS))
_runtime, _info = MEASUREMENTS[DEFAULT_MEASUREMENT]
_batches = [n for n in timed_batches(_runtime) if n in SHOW_BATCHES]
print_table(make_table(_runtime, _batches), _batches, _info)

model                     params M  steps     n=4096        n=1   Wass.     FID    APD
RSGM 512x4                    0.91    100       54.4    27602.3   16.46  0.0125  14.85
RSGM 512x4 full               0.91    100       54.5    27732.0   16.40  0.0124  14.94
RSGM 512x4, 300               0.91    300      162.9    78923.4   16.26  0.0124  14.74
PoseD-Flow, mid 100           0.00    100      171.5    79245.4   16.63  0.0142  15.56
DPoser c10, 100               7.22    100      357.9    43832.3   16.35  0.0138  14.80
DPoser full, 100              7.22    100      359.1    43675.9   16.31  0.0117  15.11
RSGM 1536x6                  12.16    100      468.8    36731.7   16.42  0.0115  15.16
RSGM 512x4, 1000              0.91   1000      541.3   243483.0   16.15  0.0129  14.62
DPoser c10, 1000              7.22   1000     3582.9   392159.1   16.21  0.0122  14.72
DPoser full, 1000             7.22   1000     3610.1   395924.1   16.21  0.0114  15.11
RSGM 4096x7                 101.63    100  

## 5. The Pareto fronts

A model is Pareto-optimal if no other model is both faster and at least as good on the metric. Above the plots: the
measurement (machine), and checkboxes for the n of `sample(n)` to draw (the largest n filled markers, the others open);
a front is drawn for each n. The models of `FRONT_EXCLUDE` (diamonds) are drawn but do not take part in the fronts.
The button saves the figure, on a white background, to the path in the field (the format by the extension: .png,
.pdf, .svg).

In [6]:
def pareto(rows, metric, lower_better, n):
    """The rows timed at n on the front, from the fastest: each is better on `metric` than every faster one."""
    sign = 1 if lower_better else -1
    front, best = [], np.inf
    for r in sorted((r for r in rows if n in r["times"]), key=lambda r: (r["times"][n], sign * r[metric])):
        if sign * r[metric] < best:
            front.append(r)
            best = sign * r[metric]
    return front


FILL = ["full", "none", "bottom", "left", "right", "top"]  # the marker of the k-th largest n
DASHES = ["--", ":", "-.", (0, (5, 2, 1, 2)), (0, (1, 3))]  # the front of the k-th largest n
FRONT_COLOR = "0.25"


def figure(stage, batches):
    """The Pareto plots of one measurement at the n of `batches` (the largest first), on white."""
    runtime, info = MEASUREMENTS[stage]
    rows = make_table(runtime, batches)
    front_rows = [r for r in rows if r["name"] not in FRONT_EXCLUDE]
    with plt.style.context("default"):  # dark or light theme of the notebook: the figure stays white
        fig, axes = plt.subplots(1, len(METRICS), figsize=(6 * len(METRICS), 5.5), sharey=True)
        for ax, (metric, (_, unit, lower_better)) in zip(axes, METRICS.items()):
            ax.axvline(REAL[metric], color="0.6", lw=1, ls=":", zorder=0)
            ax.annotate("real train poses", (REAL[metric], 1), xycoords=("data", "axes fraction"), xytext=(3, -12),
                        textcoords="offset points", fontsize=8, color="0.4")
            for k, n in enumerate(batches):
                front = pareto(front_rows, metric, lower_better, n)
                ax.plot([r[metric] for r in front], [r["times"][n] for r in front], ls=DASHES[k % len(DASHES)], lw=1.4,
                        color=FRONT_COLOR, zorder=1)
                for r in rows:
                    if n not in r["times"]:
                        continue
                    c, marker = FAMILIES[r["family"]], "D" if r["name"] in FRONT_EXCLUDE else "o"
                    ax.plot(r[metric], r["times"][n], marker=marker, ms=8 if k == 0 else 7, ls="", color=c,
                            fillstyle=FILL[k % len(FILL)], markerfacecoloralt="white", markeredgecolor=c if k else "white",
                            markeredgewidth=1.2, zorder=3 if k == 0 else 2)
                    if k == 0:  # the labels only at the largest n: the others are above or below the same x
                        ax.annotate(r["short"], (r[metric], r["times"][n]), xytext=(5, 4), textcoords="offset points",
                                    fontsize=7.5, color="0.15")
            ax.set_yscale("log")
            if not lower_better:
                ax.invert_xaxis()  # the better side on the left, as GRACE
            ax.set_xlabel(f"← {metric}" + (f" ({unit})" if unit else "") + ("" if lower_better else ", higher is better"))
            ax.grid(True, which="major", color="0.92", lw=0.8)
            for side in ("top", "right"):
                ax.spines[side].set_visible(False)
        axes[0].set_ylabel("↓ time per pose (µs)")
        handles = [plt.Line2D([], [], marker="o", ls="", color=c, markeredgecolor="white", markersize=8, label=f)
                   for f, c in FAMILIES.items()]
        handles += [plt.Line2D([], [], marker="o", ls="", color="0.4", fillstyle=FILL[k % len(FILL)], markerfacecoloralt="white",
                               markersize=8, label=f"n = {n}") for k, n in enumerate(batches)]
        handles += [plt.Line2D([], [], marker="D", ls="", color="0.4", markersize=7, label="not in the fronts (†)")]
        handles += [plt.Line2D([], [], ls=DASHES[k % len(DASHES)], color=FRONT_COLOR, label=f"Pareto front, n = {n}")
                    for k, n in enumerate(batches)]
        fig.legend(handles=handles, loc="lower center", ncol=min(len(handles), 10), frameon=False, fontsize=9)
        fig.suptitle("Sample quality against the cost of sampling; † trained on our AMASS", fontsize=11, y=0.995)
        fig.text(0.5, 0.945, machine(info), ha="center", fontsize=8.5, color="0.3")
        fig.tight_layout(rect=(0, 0.08 if len(handles) > 10 else 0.06, 1, 0.94))
        fig.patch.set_facecolor("white")
    return fig, rows


def default_path(stage, batches):
    return str(FIGURES / f"pareto_{stage}_n{'_'.join(map(str, batches))}.png")


# ---- the controls: measurement, n, save ----
measurement = widgets.Dropdown(
    options=[(f"{', '.join(info['chip'])} ({', '.join(info['device'])}): {len(rt)} models", stage)
             for stage, (rt, info) in MEASUREMENTS.items()],
    value=DEFAULT_MEASUREMENT, description="measurement", layout=widgets.Layout(width="520px"),
    style={"description_width": "initial"})
batch_boxes = widgets.HBox()
path_field = widgets.Text(description="save to", layout=widgets.Layout(width="760px"), style={"description_width": "initial"})
save_button = widgets.Button(description="save", icon="save", layout=widgets.Layout(width="90px"))
status = widgets.Label()
out = widgets.Output()
state = {"fig": None, "default": None}


def checked():
    return [box.n for box in batch_boxes.children if box.value]


def redraw(*_):
    stage, batches = measurement.value, checked()
    if path_field.value in ("", state["default"]):  # follow the choice, unless a path was typed
        path_field.value = state["default"] = default_path(stage, batches)
    status.value = ""
    with out:
        out.clear_output(wait=True)
        runtime, info = MEASUREMENTS[stage]
        if not batches:
            print("no n checked")
            state["fig"] = None
            return
        fig, rows = figure(stage, batches)
        print_table(rows, batches, info)
        display(fig)
        plt.close(fig)
        state["fig"] = fig
        for n in batches:
            for metric, (_, _, lower_better) in METRICS.items():
                print(f"n = {n:<5d} {metric:12s} front: " + " → ".join(r["short"] for r in pareto(
                    [r for r in rows if r["name"] not in FRONT_EXCLUDE], metric, lower_better, n)))


def set_batches(*_):
    """Checkboxes for the n of the chosen measurement; those of SHOW_BATCHES checked (else the largest)."""
    available = timed_batches(MEASUREMENTS[measurement.value][0])
    chosen = [n for n in available if n in SHOW_BATCHES] or available[:1]
    boxes = []
    for n in available:
        box = widgets.Checkbox(value=n in chosen, description=f"n = {n}", indent=False, layout=widgets.Layout(width="110px"))
        box.n = n
        box.observe(redraw, names="value")
        boxes.append(box)
    batch_boxes.children = boxes
    redraw()


def save(_):
    if state["fig"] is None:
        status.value = "nothing to save"
        return
    target = Path(path_field.value).expanduser()
    target.parent.mkdir(parents=True, exist_ok=True)
    state["fig"].savefig(target, dpi=200, facecolor="white", bbox_inches="tight")
    status.value = f"saved {target}"


measurement.observe(set_batches, names="value")
save_button.on_click(save)
display(widgets.VBox([measurement, batch_boxes, widgets.HBox([path_field, save_button, status]), out]))
set_batches()

## 6. Reading (proof of concept)

- Every model is one point at its own sampler: the steps are not swept yet, except RSGM 512x4 (100 / 300 / 1000) and
  DPoser (100 / 1000). A sweep of the steps (5–200) for RSGM, PoseD-Flow and DPoser would turn each into a curve.
- The differences in quality are small against the noise: the ± of the metrics (in `posedflow_vs_rsgm.ipynb`) is of
  the order of the gaps between the points. The front on one metric is therefore not robust yet.
- † models are trained on other data than they are evaluated on; their quality is a lower bound.
- The times depend on this machine; on a GPU the ratio of the flow to the ODE/SDE models may differ.